In [ ]:
"""
pubmed_pathway_serous_association.py

Given a list of pathway names, query PubMed to find:
 - number of papers mentioning each pathway
 - number of papers mentioning serous ovarian cancer
 - number of papers mentioning both
 - compute odds ratio (Fisher exact) and p-value
 - adjust p-values (Benjamini-Hochberg)
 - write CSV and print summary

Requirements:
  - biopython (Entrez)
  - pandas
  - scipy
  - statsmodels
  - tqdm

Usage:
  - edit the PATHWAYS list or load from a file and run.
  - set Entrez.email (required) and Entrez.api_key (optional)
"""

from Bio import Entrez
import time
import pandas as pd
from scipy.stats import fisher_exact
import math
from statsmodels.stats.multitest import multipletests
from tqdm import tqdm

# --------- USER CONFIGURATION ----------
Entrez.email = "e8124011@sriher.edu.in"   # <<-- REPLACE with your email (required by NCBI)
# Entrez.api_key = "8b1b23489175f038f7ac6dac2324dee44008"    # <<-- optional but recommended

# Delay between consecutive Entrez requests (seconds). Respect NCBI rate limits.
# If you do NOT supply api_key, keep delay >= 0.34 (3 requests/sec) or larger.
DELAY = 0.35

# Pathways to test: replace/extend this list or load from a file.
PATHWAYS = [
    "PI3K-Akt signaling pathway",
    "MAPK signaling pathway",
    "Wnt signaling pathway",
    "Notch signaling pathway",
    "TGF-beta signaling pathway"
]

# Synonyms / search term variants for 'serous ovarian cancer' (common).
SEROUS_TERMS = [
    '"serous ovarian cancer"',
    '"serous ovarian carcinoma"',
    '"high-grade serous ovarian cancer"',
    '"high grade serous ovarian cancer"',
    '"HGSC"',
    '"serous carcinoma of the ovary"',
    '"serous ovarian neoplasm"'
]
# Combine synonyms into an OR clause for PubMed
SEROUS_QUERY = "(" + " OR ".join(SEROUS_TERMS) + ")"

# Whether to use PubMed title/abstract only or full text mentions.
# Default uses Title/Abstract search fields to reduce false positives:
# for pathway query we will search pathway_term[Title/Abstract]
SEARCH_FIELD = "[Title/Abstract]"

# Output CSV file
OUTPUT_CSV = "pathway_serous_pubmed_association.csv"
# --------------------------------------


def esearch_count(term):
    """Return count of PubMed records matching 'term'."""
    # Use retmax=0 just to get the count
    handle = Entrez.esearch(db="pubmed", term=term, retmax=0)
    result = Entrez.read(handle)
    handle.close()
    count = int(result.get("Count", "0"))
    time.sleep(DELAY)
    return count


def build_pathway_query(pathway):
    """
    Build a PubMed query for a pathway.
    - default: search pathway phrase in Title/Abstract.
    - if pathway contains special characters, wrap in quotes.
    """
    p = pathway.strip()
    # Quote multi-word terms
    if " " in p and not (p.startswith('"') and p.endswith('"')):
        p_quoted = f'"{p}"'
    else:
        p_quoted = p
    return f"{p_quoted}{SEARCH_FIELD}"


def main(pathways):
    # 1) Get count for serous ovarian cancer overall
    print("Counting overall serous ovarian cancer articles on PubMed...")
    serous_count = esearch_count(SEROUS_QUERY)
    print(f"Serous ovarian cancer articles (any of synonyms): {serous_count}")

    # 2) Optionally get total PubMed count (universe). We'll fetch total article count
    # by querying PubMed for a very broad term like "pubmed[filter]" is not reliable.
    # Instead we use Entrez.einfo to get current Count for pubmed database (approx total).
    print("Fetching approximate total PubMed article count (database size)...")
    einfo = Entrez.einfo(db="pubmed")
    einfo_data = Entrez.read(einfo)
    einfo.close()
    total_pubmed = int(einfo_data["DbInfo"]["Count"])
    print(f"Approx total PubMed records: {total_pubmed}")

    rows = []
    # iterate pathways with progress bar
    for pathway in tqdm(pathways, desc="Pathways"):
        # build pathway query
        pq = build_pathway_query(pathway)
        # count papers mentioning pathway
        try:
            pathway_count = esearch_count(pq)
        except Exception as e:
            print(f"Error counting pathway '{pathway}': {e}")
            pathway_count = 0

        # count papers mentioning both pathway and serous ovarian cancer
        combined_query = f"({pq}) AND {SEROUS_QUERY}"
        try:
            both_count = esearch_count(combined_query)
        except Exception as e:
            print(f"Error counting both for '{pathway}': {e}")
            both_count = 0

        # Build contingency table:
        # A = both_count
        A = both_count
        # B = pathway only (pathway_count - both)
        B = max(0, pathway_count - both_count)
        # C = serous only (serous_count - both)
        C = max(0, serous_count - both_count)
        # D = neither = total_pubmed - (A+B+C)
        D = max(0, total_pubmed - (A + B + C))

        # If any cell is zero, fisher_exact can still compute but oddsratio may be inf/0.
        # We'll add standard 0.5 continuity correction for calculation of odds ratio when needed.
        table_for_test = [[A, B], [C, D]]
        # If any value is zero, add 0.5 to all cells for OR computation only (but use fisher for p-value on raw table)
        table_for_or = None
        if 0 in (A, B, C, D):
            table_for_or = [[A + 0.5, B + 0.5], [C + 0.5, D + 0.5]]
        else:
            table_for_or = table_for_test

        # Compute odds ratio (from table_for_or) and Fisher exact p-value (raw table)
        try:
            # fisher_exact returns (oddsratio, pvalue)
            # Use the raw integer table for Fisher's test p-value (recommended)
            _, p_value = fisher_exact(table_for_test, alternative='two-sided')
        except Exception as e:
            print(f"Fisher exact error for {pathway}: {e}")
            p_value = 1.0

        # Compute odds ratio from corrected table using formula (A*D)/(B*C)
        try:
            or_value = (table_for_or[0][0] * table_for_or[1][1]) / (table_for_or[0][1] * table_for_or[1][0])
        except Exception as e:
            or_value = float('nan')

        rows.append({
            "pathway": pathway,
            "pathway_count": pathway_count,
            "serous_count": serous_count,
            "both_count": both_count,
            "A": A, "B": B, "C": C, "D": D,
            "odds_ratio": or_value,
            "p_value": p_value
        })

    df = pd.DataFrame(rows)
    # Multiple testing correction (Benjamini-Hochberg)
    pvals = df["p_value"].values
    reject, pvals_corrected, alphacSidak, alphacBonf = multipletests(pvals, alpha=0.05, method='fdr_bh')
    df["p_adjusted_BH"] = pvals_corrected
    df["significant_BH"] = reject

    # Sort by p_adjusted
    df = df.sort_values("p_adjusted_BH")

    # Save to CSV
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Results saved to {OUTPUT_CSV}")

    # Print a quick summary table
    print("\nTop associations by adjusted p-value:")
    display_cols = ["pathway", "pathway_count", "both_count", "odds_ratio", "p_value", "p_adjusted_BH", "significant_BH"]
    print(df[display_cols].head(20).to_string(index=False))

    return df


if __name__ == "__main__":
    # If you want to read pathways from a file, replace PATHWAYS with reading logic.
    df_out = main(PATHWAYS)
    # If you want the DataFrame available for further inspection in an interactive session, keep df_out.


Counting overall serous ovarian cancer articles on PubMed...
Serous ovarian cancer articles (any of synonyms): 5340
Fetching approximate total PubMed article count (database size)...
Approx total PubMed records: 40120456


Pathways: 100%|██████████| 5/5 [00:07<00:00,  1.42s/it]

Results saved to pathway_serous_pubmed_association.csv

Top associations by adjusted p-value:
                   pathway  pathway_count  both_count  odds_ratio      p_value  p_adjusted_BH  significant_BH
    MAPK signaling pathway           8984          12   10.067895 5.844541e-09   2.922271e-08            True
PI3K-Akt signaling pathway          10767          12    8.398432 4.135861e-08   1.033965e-07            True
     Wnt signaling pathway           7906           8    7.619128 1.461985e-05   2.436642e-05            True
   Notch signaling pathway           3972           5    9.476293 2.213801e-04   2.767251e-04            True
TGF-beta signaling pathway           2803           1    2.681326 3.114139e-01   3.114139e-01           False


In [ ]:
!pip install biopython


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.8 MB/s eta 0:00:00


In [ ]:
!pip install biopython pandas scipy statsmodels tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import time
from Bio import Entrez
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
from tqdm import tqdm
Entrez.email = "e8124011@sriher.edu.in"
Entrez.api_key = "api_key"
DELAY = 0.35
INPUT_FILE = "/content/Reactome_Pathways_2024_table (5).txt"
OUTPUT_FILE = "Serous_Ovarian_Pathway_PubMed_Association_1.csv"
# STEP 1: Load pathway file
df_input = pd.read_csv(INPUT_FILE, sep="\t")
pathways = df_input["Term"].dropna().unique().tolist()
print(f"Total pathways loaded: {len(pathways)}")
# STEP 2: Define disease query
SEROUS_QUERY = """
(
"serous ovarian cancer"[Title/Abstract] OR
"serous ovarian carcinoma"[Title/Abstract] OR
"high-grade serous ovarian cancer"[Title/Abstract] OR
"HGSC"[Title/Abstract]
)
AND
(
pathogenesis[Title/Abstract] OR
mechanism[Title/Abstract]
)
AND
humans[MeSH Terms]
"""

# ===============================
# Helper function
# ===============================
def pubmed_count(query):
    handle = Entrez.esearch(db="pubmed", term=query, retmax=0)
    result = Entrez.read(handle)
    handle.close()
    time.sleep(DELAY)
    return int(result["Count"])

# ===============================
# STEP 3: Get background counts
# ===============================
print("Fetching total PubMed count...")
einfo = Entrez.einfo(db="pubmed")
einfo_data = Entrez.read(einfo)
einfo.close()
TOTAL_PUBMED = int(einfo_data["DbInfo"]["Count"])

print("Fetching serous ovarian cancer count...")
serous_count = pubmed_count(SEROUS_QUERY)

# ===============================
# STEP 4: Pathway loop
# ===============================
results = []

for pathway in tqdm(pathways):

    pathway_query = f'"{pathway}"[Title/Abstract]'
    combined_query = f'({pathway_query}) AND ({SEROUS_QUERY})'

    try:
        pathway_count = pubmed_count(pathway_query)
        both_count = pubmed_count(combined_query)
    except:
        pathway_count = 0
        both_count = 0

    A = both_count
    B = pathway_count - both_count
    C = serous_count - both_count
    D = TOTAL_PUBMED - (A + B + C)

    if min(A, B, C, D) == 0:
        A += 0.5; B += 0.5; C += 0.5; D += 0.5

    odds_ratio = (A * D) / (B * C)
    _, p_value = fisher_exact([[A, B], [C, D]])

    results.append({
        "Pathway": pathway,
        "Pathway_Papers": pathway_count,
        "Serous_OC_Papers": serous_count,
        "Overlap_Papers": both_count,
        "Odds_Ratio": odds_ratio,
        "P_value": p_value
    })

# ===============================
# STEP 5: Multiple Testing Correction
# ===============================
results_df = pd.DataFrame(results)

adjusted = multipletests(results_df["P_value"], method="fdr_bh")
results_df["Adjusted_P_value"] = adjusted[1]
results_df["Significant"] = adjusted[0]

# Keep only significant pathways
significant_df = results_df[results_df["Significant"] == True]
significant_df = significant_df.sort_values("Adjusted_P_value")

# Save
significant_df.to_csv(OUTPUT_FILE, index=False)

print("\nAnalysis complete.")
print(f"Significant pathways found: {len(significant_df)}")
print(f"Results saved to: {OUTPUT_FILE}")
print("\nTop significant pathways:")
print(significant_df.head())


Total pathways loaded: 905
Fetching total PubMed count...
Fetching serous ovarian cancer count...


100%|██████████| 905/905 [14:44<00:00,  1.02it/s]


Analysis complete.
Significant pathways found: 8
Results saved to: Serous_Ovarian_Pathway_PubMed_Association_1.csv

Top significant pathways:
        Pathway  Pathway_Papers  Serous_OC_Papers  Overlap_Papers  Odds_Ratio  \
459   Apoptosis          533080               377              47   10.577497   
223  DNA Repair           59334               377              17   31.892345   
44   Cell Cycle          212183               377              24   12.788929   
95      Disease         4569420               377             105    3.003435   
816   Autophagy           97011               377              10   11.242638   

          P_value  Adjusted_P_value  Significant  
459  1.999325e-30      1.809389e-27         True  
223  5.719847e-20      2.588231e-17         True  
44   1.974062e-18      5.955086e-16         True  
95   2.947997e-18      6.669844e-16         True  
816  4.327488e-08      7.832754e-06         True  
